# Week 6 — Final Inference: Single Image Prediction + Grad-CAM

**Project:** Can We Trust AI-Image Detectors on Ugandan Content?
**Member:** AW
**Goal:** Classify ONE unseen image with label + confidence score + Grad-CAM heat map.
This is the end-to-end inference pipeline — what a fact-checker would actually use.

### What it outputs
- Label: real / AI-generated
- Confidence score
- Grad-CAM heat map visualization
- JSON result file
- On-screen disclaimer: screening aid, not proof of authenticity

### RAM-safe design
- 128×128 image
- Loads saved model from `reports/`
- Run cells one at a time

### Drive paths
- Root: `/content/drive/MyDrive/Deepfake Detection Project/datasets`
- Model: `reports/week3_run_c_aug.pt` (or Week 2 fallback)
- Outputs: `reports/prediction_*.png`, `reports/prediction_*.json`

In [ ]:
!pip -q install torch torchvision pillow

In [ ]:
import os, sys, json, gc
from datetime import datetime

import numpy as np
from PIL import Image as PILImage

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

print('PyTorch:', torch.__version__)
print('Imports OK')

## 1. Mount Drive + Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

ROOT = '/content/drive/MyDrive/Deepfake Detection Project/datasets'
MEMBER = 'AW'

IMG_SIZE = 128

# --- SET THIS TO THE IMAGE YOU WANT TO CLASSIFY ---
# Example: IMAGE_PATH = os.path.join(ROOT, 'pilot_data', 'ai', 'ai_0000.jpg')
IMAGE_PATH = None  # change me before running

np.random.seed(42)
torch.manual_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

for d in ['reports', 'metadata', 'audit_outputs']:
    os.makedirs(os.path.join(ROOT, d), exist_ok=True)

print('Drive root:', ROOT)
print('Set IMAGE_PATH in the cell above before running inference.')

## 2. TinyCNN model + Grad-CAM

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self, img_size=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x.squeeze(1)


class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.handles = []
        
        def fwd_hook(module, input, output):
            self.activations = output.detach()
        
        def bwd_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()
        
        self.handles.append(target_layer.register_forward_hook(fwd_hook))
        self.handles.append(target_layer.register_full_backward_hook(bwd_hook))
    
    def __call__(self, x, class_idx=1):
        self.model.zero_grad()
        logits = self.model(x)
        score = logits[:, class_idx] if logits.dim() > 1 else logits[class_idx]
        score.backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=(x.shape[2], x.shape[3]), mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        if cam.max() > cam.min():
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        return cam, logits.detach().cpu().numpy()
    
    def close(self):
        for h in self.handles:
            h.remove()


# Load model
model = TinyCNN(img_size=IMG_SIZE)
model_path = os.path.join(ROOT, 'reports', 'week3_run_c_aug.pt')
if not os.path.exists(model_path):
    model_path = os.path.join(ROOT, 'reports', 'tinycnn_baseline.pt')
    print('Falling back to Week 2 baseline')

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    print(f'Loaded model from {model_path}')
else:
    print('WARNING: No saved model found — run Week 2 or Week 3 first')

model = model.to(DEVICE)
model.eval()

# Grad-CAM on last conv layer
gradcam = GradCAM(model, model.features[-4])

## 3. Run inference on one image

Set `IMAGE_PATH` above, then run this cell.

In [ ]:
if IMAGE_PATH is None:
    print('ERROR: Set IMAGE_PATH in the config cell above.')
    print('Example: IMAGE_PATH = os.path.join(ROOT, \'pilot_data\', \'ai\', \'ai_0000.jpg\')')
else:
    if not os.path.exists(IMAGE_PATH):
        print(f'ERROR: Image not found: {IMAGE_PATH}')
    else:
        # Preprocess
        to_tensor = transforms.ToTensor()
        normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                         std=[0.229, 0.224, 0.225])
        
        with PILImage.open(IMAGE_PATH) as im:
            im = im.convert('RGB').resize((IMG_SIZE, IMG_SIZE))
            img_arr = np.array(im)
            x = normalize(to_tensor(im)).unsqueeze(0).to(DEVICE)
        
        # Predict
        with torch.no_grad():
            logits = model(x)
            prob_ai = float(torch.sigmoid(logits).cpu().numpy()[0][0])
        
        label = 'ai' if prob_ai >= 0.5 else 'real'
        confidence = prob_ai if label == 'ai' else 1 - prob_ai
        
        # Grad-CAM
        cam, _ = gradcam(x)
        gradcam.close()
        
        # Print result
        print('=' * 60)
        print('SCREENING RESULT')
        print('=' * 60)
        print(f'Image: {IMAGE_PATH}')
        print(f'Label: {label.upper()}')
        print(f'Confidence: {confidence:.4f} (P(AI)={prob_ai:.4f})')
        print()
        print('DISCLAIMER: This is a screening aid, NOT proof of authenticity.')
        print('A human fact-checker should review any suspicious image.')
        print('The model does not accuse or clear anyone.')
        print('=' * 60)
        
        # Save visualization
        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        axes[0].imshow(img_arr)
        axes[0].set_title('Original')
        axes[0].axis('off')
        
        axes[1].imshow(cam, cmap='jet')
        axes[1].set_title('Grad-CAM heat map')
        axes[1].axis('off')
        
        axes[2].imshow(img_arr)
        axes[2].imshow(cam, cmap='jet', alpha=0.5)
        axes[2].set_title('Overlay')
        axes[2].axis('off')
        
        fig.suptitle(f'Prediction: {label.upper()}  |  Confidence: {confidence:.3f}',
                     fontsize=14, color='red' if label == 'ai' else 'green')
        
        base = os.path.splitext(os.path.basename(IMAGE_PATH))[0]
        viz_path = os.path.join(ROOT, 'reports', f'prediction_{base}.png')
        fig.savefig(viz_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        
        # Save JSON
        result = {
            'week': 6,
            'member': MEMBER,
            'image': IMAGE_PATH,
            'label': label,
            'confidence': float(confidence),
            'p_ai': float(prob_ai),
            'model': model_path,
            'disclaimer': 'Screening aid, not proof of authenticity. Human review required.',
            'timestamp': datetime.utcnow().isoformat() + 'Z',
        }
        json_path = os.path.join(ROOT, 'reports', f'prediction_{base}.json')
        with open(json_path, 'w') as f:
            json.dump(result, f, indent=2)
        
        print(f'\nVisualization saved to {viz_path}')
        print(f'Result saved to {json_path}')
        gc.collect()